In [1]:
import pandas as pd;
import numpy as np;
 

# DOWNLOAD FILE FIRST: https://syd1.digitaloceanspaces.com/duckgoesmeow/bushfire-data/data.csv if you're interested to run this script
FILE_PATH = "INSERT DATASET PATH HERE"

bushfire_df = pd.read_csv(FILE_PATH, low_memory=False)

columns_of_interest = ['FOD_ID','DISCOVERY_DATE', 'DISCOVERY_TIME', 'DISCOVERY_DOY', 'NWCG_GENERAL_CAUSE', 'FIRE_SIZE', 'FIRE_SIZE_CLASS', 'LATITUDE' , 'LONGITUDE' , 'STATE' ]
renamed_columns = ["fire_id", "discovery_date", "discovery_time", "discovery_doy", "general_cause", "fire_size", "fire_class", "latitude", "longitude", "state"]

# Rename the columns for better conventions:
bushfire_df.rename(columns={
                         'FOD_ID':'fire_id',
                         'DISCOVERY_DATE':'discovery_date',
                         'DISCOVERY_DOY': 'discovery_doy',
                         'DISCOVERY_TIME' : 'discovery_time',
                         'NWCG_GENERAL_CAUSE':'general_cause',
                         'FIRE_SIZE' : 'fire_size',
                         'FIRE_SIZE_CLASS' : 'fire_class',
                         'LATITUDE': 'latitude',
                         'LONGITUDE':'longitude',
                         'STATE':'state'}, inplace=True)

In [2]:
# Remove unnecessary columns - keep what we need =)
bushfire_df = bushfire_df[renamed_columns]

In [3]:
# Create discovery month for easier exploration
bushfire_df.insert(loc=2, column='discovery_month', value=pd.to_datetime(bushfire_df['discovery_date']).dt.month, allow_duplicates=False)

In [4]:
# Cast discovery_date to allow for further reformating
bushfire_df['discovery_date'] = pd.to_datetime(arg=bushfire_df['discovery_date']).astype(str)

In [5]:
# Replace not-present discovery_time values and re-cast to string for further reformating
bushfire_df['discovery_time'] = pd.to_numeric(bushfire_df['discovery_time'], errors='coerce').fillna(0).astype(int).astype(str)

In [6]:
# Convert the default format for better conventions and readability
def format_time(time_str):
    if pd.isna(time_str):
        return np.nan
    time_str = str(time_str).zfill(4)
    if time_str == '0000' or time_str == '2400':
        return '00:00'
    elif 0 <= int(time_str) <= 2359:
        return f"{time_str[:2]}:{time_str[2:]}"
    else: 
        return np.nan
    
# invoke our 'format_time' function
bushfire_df['discovery_time'] = bushfire_df['discovery_time'].apply(format_time)

In [7]:
# NOTE:
# The 'origin' variable isn't really required for intial focus on Natural causes because -
# -the general_cause will be Natural by default. 
# However, if we have time available, we can modify our to include more other causes.


# Dictionary/Object to categorize the general_cause for better classication
map_cause = {'Power generation/transmission/distribution':'Accidental',
            'Natural':'Natural',
            'Debris and open burning':'Accidental',
            'Missing data/not specified/undetermined':'Missing',
            'Recreation and ceremony':'Accidental',
            'Equipment and vehicle use':'Accidental',
            'Arson/incendiarism':'Criminal',
            'Fireworks':'Accidental',
            'Other causes':'Accidental',
            'Railroad operations and maintenance':'Accidental',
            'Smoking':'Accidental',
            'Misuse of fire by a minor':'Accidental',
            'Firearms and explosives use':'Accidental'}

# create a category identifier variable 'origin' to better categorize the bushfire cause
bushfire_df.insert(loc=6, column='origin', value=bushfire_df['general_cause'].map(map_cause))

In [8]:
# create a 'season' variable for data analysis
def yieldCurrentSeason(month: int) -> str:
    if month in [12, 1, 2]:  # Winter
        return "winter"
    elif month in [3, 4, 5]:  # Spring
        return "spring"
    elif month in [6, 7, 8]:  # Summer
        return "summer"
    elif month in [9, 10, 11]:  # Autumn
        return "autumn"
    else:
        return "Invalid month"

# create 'season' variable and invoke 'yieldCurrentSeason'
bushfire_df.insert(loc=12, column='season', value=bushfire_df['discovery_month'].apply(yieldCurrentSeason))

In [9]:
# NOTE:
# For now, let's only focus on predicting natural causes and if time allows, we can tune our model to consider other human related origins causes.
bushfire_df.drop(bushfire_df.loc[bushfire_df['origin'] != 'Natural'].index, inplace=True)

In [10]:
# Finally, drop all rows with a discovery_date less than '2000-01-01'
bushfire_df.drop(bushfire_df.loc[bushfire_df['discovery_date'] < '2000-01-01'].index, inplace=True)

In [11]:
output_file = './cleaned-bushfire-data.csv'
bushfire_df.to_csv(output_file, index=False, encoding='utf-8')

In [12]:
# Let's test and see if our final results work
cleaned_df = pd.read_csv('./cleaned-bushfire-data.csv')
cleaned_df.sample(12)

,fire_id,discovery_date,discovery_month,discovery_time,discovery_doy,general_cause,origin,fire_size,fire_class,latitude,longitude,state,season
172745,201775875,2013-08-20,8,09:55,232,Natural,Natural,0.1,A,36.693300,-107.517500,NM,summer
117645,1024123,2000-03-20,3,14:34,80,Natural,Natural,1.0,B,34.645000,-95.488300,OK,spring
230545,400603767,2020-06-18,6,18:35,170,Natural,Natural,2.0,B,43.222115,-77.220143,NY,summer
224883,400509369,2019-09-08,9,11:00,251,Natural,Natural,0.1,A,47.985600,-102.483900,ND,autumn
93492,430512,2007-03-19,3,13:34,78,Natural,Natural,0.8,B,25.712780,-80.463610,FL,spring
183030,300027050,2014-07-15,7,20:53,196,Natural,Natural,0.1,A,38.612360,-109.393080,UT,summer
177287,300000035,2014-08-13,8,20:20,225,Natural,Natural,0.5,B,46.048056,-115.293611,ID,summer
93795,431525,2007-06-19,6,16:30,170,Natural,Natural,0.1,A,28.980830,-81.118060,FL,summer
168474,201762787,2013-07-11,7,10:09,192,Natural,Natural,0.1,A,34.095556,-109.499722,AZ,summer
188243,300203863,2015-09-24,9,20:00,267,Natural,Natural,0.1,A,43.665278,-103.828611,SD,autumn
